In [67]:
# Imports
import pandas as pd
import numpy as np
from tabulate import tabulate
from sklearn.model_selection import train_test_split
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error

## Utility Functions

In [26]:
def load_node_profile(node):
    filepath = f'../../tool/infrastructure-profiler/profiles/{node}.csv'

    with open(filepath) as f:
        data = f.readlines()[1].split(',')

    return {
        'ram': int(data[1].strip()),
        'cores': int(data[2].strip()),
        'io_read': float(data[3].strip()),
        'io_write': float(data[4].strip()),
        'sysbench': float(data[5].strip()),
        'z7b': float(data[6].strip()),
        'bench_time': float(data[7].strip())
    }

def combine_df(input_df, trace_df):
    rows = []
    # assert len(trace_df) == len(input_df['name'].unique())

    for name, group in input_df.groupby('name'):
        no_files = str(len(group))
        filepaths = group['path']
        paths = ','.join([filepath.split('/')[-1].split('.')[-1] for filepath in filepaths])
        total_size = group['size'].sum() / 1073741824  # convert from B to GiB
        task_record = trace_df[trace_df['name'] == name].iloc[0]
        row = [task_record['task_id'], task_record['name'], task_record['%cpu'], 
               task_record['start'], task_record['complete'], task_record['complete'] - task_record['start'], 
               no_files, paths, total_size, task_record['name'].split(':')[-1].split(' ')[0]]
        rows.append(row)

    combined = pd.DataFrame(rows, columns=['id', 'name', '%cpu', 'start', 'complete', 'runtime', 'no_files', 'files', 'input_gib', 'short_name'])
    return combined

def load_profiling_runs(node, workflows, samples, node_profiles):
    node_profile = node_profiles[node]
    runs = {}

    for workflow, samples in zip(workflows, samples):
        runs[workflow] = {}

        for sample in samples:
            sample_dir = f'{node}/{workflow}/{sample}/'
            input_df = pd.read_csv(sample_dir + 'input.csv', sep=';')
            trace_df = pd.read_csv(sample_dir + 'trace.csv')
            runs[workflow][sample] = combine_df(input_df, trace_df)

    return (node_profile, runs)

def load_workflow_execution(cluster, workflow, iteration):
    execution_dir = f'../{cluster}/{workflow}/{iteration}/'
    input_df = pd.read_csv(execution_dir + 'input.csv', sep=';')
    trace_df = pd.read_csv(execution_dir + 'trace.csv')
    return combine_df(input_df, trace_df)

def load_power_model(node, setting=''):
    if setting != '':
        filename = f'{setting}-model.txt'
    else: 
        filename = 'model.txt'

    with open(f'../../tool/infrastructure-profiler/raw/{node}/{filename}', 'r') as f:
        data = [line.strip() for line in f.readlines()[1:]]

    coeff = float(data[0].split(',')[0][2:].strip())
    inter = float(data[0].split(',')[1][:-2].strip())
    draw = float(data[1].split(':')[1].strip())

    if draw == 0:
        draw = 0.392

    return (coeff, inter, draw)

def get_task_data_from_experiment(task_name, experiment):
    task_data = experiment[experiment['short_name'] == task_name]
    key_data = task_data[['input_gib', 'runtime']]
    key_data.columns = ['x', 'y']
    key_data.reset_index().drop('index', axis=1)
    return key_data

In [ ]:
class MedianModel:
    def __init__(self, median):
        self.median = median
    
    def predict(self, x):
        x = np.asarray(x)
        value = x.shape[0]
        return np.full(value, self.median)

## Setup Data

In [10]:
# Constants
HU = 'hu'
HU_40 = 'hu-c40'
GPG = 'gpg'
GPG_16 = 'gpgnode-16'
GPG_22 = 'gpgnode-22'
TU = 'tu'
TU_06 = 'tu-06'
HU_42 = 'hu-c42'
AWS = 'aws'
AWS_c5n = 'aws-metal-c5n'
AWS_z1d = 'aws-metal-z1d'
wfs = {
    GPG: ['rnaseq', 'chipseq'], 
    HU: ['rnaseq', 'chipseq', 'rangeland']
}

In [11]:
# Node Profiles + Power Models (todo)
nodes = [AWS_c5n, AWS_z1d, GPG_16, GPG_22, TU_06, HU_40, HU_42] # , 'hu-c44', 'hu-c45']
node_profiles = {}

for node in nodes:
    profile = load_node_profile(node)
    node_profiles[node] = profile

In [ ]:
# Workflow Executions 
wf_executions = {}

for cluster, cluster_path in zip([GPG, HU], ['gu-cluster', 'hu-cluster']):
    wf_executions[cluster] = {}
    for wf in wfs[cluster]:
        wf_executions[cluster][wf] = {}
        for iteration in range(1, 4):
            wf_executions[cluster][wf][iteration] = load_workflow_execution(cluster_path, wf, iteration)

In [33]:
# Load workflow profiling runs
wf_profiles = {}

# TU_06
wfs = ['rnaseq', 'chipseq']
samples = [['sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
profile, runs = load_profiling_runs(TU_06, wfs, samples, node_profiles)
wf_profiles[TU_06] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(TU_06)
}

# HU_40
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
profile, runs = load_profiling_runs(HU_40, wfs, samples, node_profiles)
wf_profiles[HU_40] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_40, 'schedutil')
}

# HU_42
wfs = ['rnaseq', 'chipseq']
samples = [['sample_a', 'sample_b', 'sample_c', 'sample_d'], ['sample_e', 'sample_f', 'sample_g', 'sample_h']]
profile, runs = load_profiling_runs(HU_42, wfs, samples, node_profiles)
wf_profiles[HU_42] = {
    'node_profile': profile,
    'executions': runs,
    'power_model': load_power_model(HU_42, 'schedutil')
}

# gpgnode-16
wf_profiles[GPG_16] = {
    'node_profile': node_profiles[GPG_16],
    'power_model': load_power_model(GPG_16, 'ondemand')
}

# changing to rnaseq a-e, chipseq f-j

## Evaluation

In [ ]:
# Prediction and Verification
def create_prediction_model(runs):
    models = {}

    for workflow in runs.keys():
        models[workflow] = {}
        samples = list(runs[workflow].keys())
        tasks = runs[workflow][samples[0]]['short_name'].unique()

        for task in tasks:
            task_data = pd.DataFrame(columns=['x', 'y'], dtype='float64')
            for sample in samples: 
                experiment_data = get_task_data_from_experiment(task, runs[workflow][sample])
                task_data = pd.concat([task_data, experiment_data])

            # only works when we actively vary the input size i.e. 4x similar sizes is useless (!) --
            # @Vasilis I'm going to do additional profiling runs with smaller samples / bigger samples to get this data so we can use these approaches :) 
            #pearson = np.corrcoef(task_data['x'].to_numpy(), task_data['y'].to_numpy())[0, 1]

            #if np.isnan(pearson) or pearson < 0.75:
            #    task_models[task] = MedianModel(np.median(task_data['y'].to_numpy()))
            #else:

            lr = LinearRegression()
            models[workflow][task] = lr.fit(task_data['x'].values.reshape(-1, 1), task_data['y'].values) 
            # alt -- Lotaru used a BayesianRidge
            #     model = BayesianRidge(fit_intercept=True)
            #     model.fit(x.reshape(-1, 1), y)

    return models

def get_translation_factor(local_profile, target_profile):
    local_write_val = local_profile['io_write'] / local_profile['ram']
    local_read_val = local_profile['io_read'] / local_profile['ram']
    target_write_val = target_profile['io_write'] / target_profile['ram']
    target_read_val = target_profile['io_read'] / target_profile['ram']
    return (0.25 * local_profile['z7b'] / target_profile['z7b']) + \
    (0.25 * local_profile['sysbench'] / target_profile['sysbench']) + \
    (0.25 * local_write_val / target_write_val) + \
    (0.25 * local_read_val / target_read_val)
    #(0.25 * local_profile['bench_time'] / target_profile['bench_time']) + \

def predict_wf_runtime(wf_model, actual_execution, factor):
    # todo print output as a table to explore
    overall_error = []
    missing_tasks = []    
    headers = ['task', 'input (GiB)', 'pred_time (s)', 'actual_time (s)', 'error (%)', 'pred_usage (%)', 'actual_usage (%)']
    data = []

    for _, record in actual_execution.iterrows():
        task_name = record['short_name']
        data_size = record['input_gib']
        actual_runtime = record['runtime']

        if task_name in wf_model.keys():
            predicted_runtime = wf_model[task_name].predict([[data_size]])[0] * factor  # todo predict the cpu_usage of a task, consider the no. cores on target vs. local
            if actual_runtime > 0:
                if predicted_runtime > 0:
                    error = (abs(predicted_runtime - actual_runtime) / actual_runtime) * 100 
        else:
            missing_tasks.append(task_name)

        data.append([task_name, f'{data_size:.2f}', f'{predicted_runtime / 1000:.2f}', f'{actual_runtime / 1000:.2f}', error, f'{record["%cpu"]}'])

    print(tabulate(data, headers, tablefmt='orgtbl'))
    # print('avg. error', np.mean(overall_error))

def predict_wf_energy(wf_model, actual_execution, factor, power_model):
    headers = ['task', 'input (GiB)', 'pred_time (s)', 'actual_time (s)', 'error (%)', 'pred_usage (%)', 'actual_usage (%)', 'pred_energy (kWh)', 'actual_energy (kWh)', 'error (%)']
    data = []

    print(tabulate(data, headers, tablefmt='orgtbl'))
    print(power_model)

    # predict task dynamic energy consumption
    # predict workflow total energy consumption (include static energy during times when nodes were individually active)

    # load task dynamic energy consumption (estimated using ichnos)
    # load workflow total energy consumption (estimated from RAPL energy logs -- ground truth here)

    # compare the error % for task dynamic energy consumption
    # compare the error % in total energy consumption

In [78]:
# TEST: hu_c40 to hu-cluster execution
hu_wf_models = create_prediction_model(wf_profiles[HU_40]['executions'])
factor = get_translation_factor(wf_profiles[HU_40]['node_profile'], wf_profiles[HU_40]['node_profile'])
target_power_model = wf_profiles[HU_40]['power_model']
hu_rnaseq_model = hu_wf_models['rnaseq']
actual_execution = wf_executions[HU]['rnaseq'][1]
predict_wf_runtime(hu_rnaseq_model, actual_execution, factor)
predict_wf_energy(hu_rnaseq_model, actual_execution, factor, target_power_model)

| task                         |   input (GiB) |   pred_time (s) |   actual_time (s) |    error (%) |   actual_usage (%) |
|------------------------------+---------------+-----------------+-------------------+--------------+--------------------|
| SAMTOOLS_FLAGSTAT            |          1.58 |           14.95 |                14 |   6.75332    |              132.8 |
| SAMTOOLS_FLAGSTAT            |          1.58 |           14.95 |                14 |   6.75364    |              133.5 |
| SAMTOOLS_FLAGSTAT            |          1.58 |           14.95 |                16 |   6.59247    |              133.5 |
| SAMTOOLS_FLAGSTAT            |          1.65 |           15.57 |                14 |  11.2041     |              134.4 |
| SAMTOOLS_FLAGSTAT            |          1.12 |           10.61 |                10 |   6.10552    |              130.4 |
| SAMTOOLS_FLAGSTAT            |          1.39 |           13.15 |                13 |   1.18962    |              133.2 |
| SAMTOOLS_FLAGS

In [81]:
# TEST: hu_c40 model to gu-cluster execution
factor = get_translation_factor(wf_profiles[HU_40]['node_profile'], wf_profiles[GPG_16]['node_profile'])
target_power_model = wf_profiles[GPG_16]['power_model']
hu_rnaseq_model = hu_wf_models['rnaseq']
actual_execution = wf_executions[GPG]['rnaseq'][1]
predict_wf_runtime(hu_rnaseq_model, actual_execution, factor)
predict_wf_energy(hu_rnaseq_model, actual_execution, factor, target_power_model)

| task                     |   input (GiB) |   pred_time (s) |   actual_time (s) |     error (%) |   actual_usage (%) |
|--------------------------+---------------+-----------------+-------------------+---------------+--------------------|
| CUSTOM_GETCHROMSIZES     |          2.93 |           23.25 |                19 |    22.3623    |               60.9 |
| GTF_FILTER               |          3.78 |           42.44 |                30 |    41.4652    |               79.4 |
| MAKE_TRANSCRIPTS_FASTA   |          3.71 |          121.04 |                96 |    26.0805    |               76.3 |
| SALMON_INDEX             |          3.2  |         4257.93 |              3235 |    31.6206    |              397.3 |
| SAMTOOLS_FLAGSTAT        |          1.38 |           35.26 |                18 |    95.9088    |              118.4 |
| SAMTOOLS_FLAGSTAT        |          1.11 |           28.45 |                16 |    77.8018    |              109.9 |
| SAMTOOLS_FLAGSTAT        |          1.

In [ ]:
# THE PLAN

# also - look at tasks that don't scale linearly (pearsson)
# investigate tasks with prediction of negative runtime?
# investigate tasks with runtime of 0

# prediction error for runtime should be ~15%, not up at 40/50%

# confirm on whether comparing dynamic power consumption for individual tasks, and then overall should consider static over periods of active execution (using traces)

# identify the governor and compare energy estimation for each workflow using ichnos (!)
# have these task-level estimates of power consumed, and total inc. and exc. static power consumption for the nodes
# automate the process

# next step, combine the predicted runtime with estimation of energy using node power models
# start to compare task-level estimates with traces


In [ ]:
# Make a simple svm, linear regression and random forest regression model to predict y from x

# try each in the comparison

# models = {
#     'SVR': SVR(),
#     'RandomForest': RandomForestRegressor(),
#     'LinearRegression': LinearRegression(),
#     'BayesianRidge': BayesianRidge()
# }

# mse = mean_squared_error(y_test, y_pred) 
# print(f'Linear Regression MSE: {mse}')
